In the EDA phase we understood that our data can be clustered pretty easily thanks to a graph obtained using t-sne dimensionality reduction algorithm to reduce our inital 20531 # of features to just 2. Now, we must preprocess our data to fit some models. We are going to try Logistic Regression, SVC and a Random Forest Classifier.

In [1]:
import pandas as pd
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from src import load_and_save_data, import_data, get_project_root

In [3]:
X_train, X_test, y_train, y_test = import_data()

Before starting out with the training phase, as we said before we have to preprocess our data. The first thing to do is to understand if there are any linear or non-linear relationships underlying in our dataset

In [4]:
from sklearn.manifold import Isomap
from sklearn.decomposition import PCA

iso_embedding = Isomap(n_components=2, n_neighbors=10)
pca = PCA(n_components=0.95, random_state=42)

X_train_iso = iso_embedding.fit_transform(X_train)
X_train_pca = pca.fit_transform(X_train)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

log_reg_iso = LogisticRegression(random_state=42)
svc_iso = SVC(random_state=42)

models = [log_reg_iso, svc_iso]
accuracies = []

for model in models:
    y_scores = cross_val_score(model, X_train_iso, y_train, cv=5, 
                               scoring="accuracy", n_jobs=-1)
    accuracies.append(y_scores.mean())

accuracies

[np.float64(0.9892857142857142), np.float64(0.9910714285714285)]

In [6]:
log_reg_pca = LogisticRegression(random_state=42)
svc_pca = SVC(random_state=42)

models = [log_reg_pca, svc_pca]
accuracies = []

for model in models:
    y_scores = cross_val_score(model, X_train_pca, y_train, cv=5, 
                               scoring="accuracy", n_jobs=-1)
    accuracies.append(y_scores.mean())

accuracies

[np.float64(0.9982142857142857), np.float64(0.9910714285714286)]

The difference in accuracy yielded by both linear and non-linear dimensionality reduction algorithms is very small. Logistic regression and SVC seems to perfrom slightly better with PCA (linear dimensionality reduction algorithm). Additionally, it could be a great insight evaluating the difference in terms of performance between our PCA Logistic Regression (simpler model) and a Random Forest (much complex model)

In [7]:
from sklearn.ensemble import RandomForestClassifier

random_clf_pca = RandomForestClassifier(oob_score = True, random_state=42)
random_clf_pca.fit(X_train_pca, y_train)

,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the followi

In [8]:
random_clf_pca.oob_score_

0.9482142857142857

In [9]:
models_dir = os.path.join(get_project_root(), "models")
joblib.dump(random_clf_pca, os.path.join(models_dir, "random_forest_pca"))

['C:\\Users\\admin\\OneDrive\\Desktop\\Python\\cancer_classification\\models\\random_forest_pca']

Using OOB (Out Of Bag) samples evaluation our random forest trained on the PCA reduced dataset has 94.81% accuracy. Now we could train another random forest model on the original dataset (without PCA reduction) and see what the accuracy is

In [10]:
random_clf = RandomForestClassifier(oob_score = True, random_state=42)
random_clf.fit(X_train, y_train)

,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the followi

In [11]:
random_clf.oob_score_

0.9964285714285714

In [12]:
joblib.dump(random_clf_pca, os.path.join(models_dir, "random_forest"))

['C:\\Users\\admin\\OneDrive\\Desktop\\Python\\cancer_classification\\models\\random_forest']

**Model Performance & Comparison**

In addition to the Random Forest trained on the PCA-reduced dataset, we also trained a baseline **Random Forest without any regularization** to evaluate its raw performance.

Surprisingly, this unregularized Random Forest reached an impressive **99.63% accuracy**. While this represents strong performance, there is a significant risk that the model may be **overfitting** to the training data.

**Overall Model Accuracies**

| Rank | Model | Accuracy | Accuracy (%) |
| :---: | :--- | :---: | :---: |
| **1** | PCA Logistic Regression | `0.9982` | 99.82% |
| **2** | Classic Random Forest *(Unregularized)* | `0.9963` | 99.63% |
| **3** | PCA Support Vector Classifier (SVC) | `0.9911` | 99.11% |
| **4** | PCA Random Forest | `0.9481` | 94.81% |

Our best models seems to be **PCA** Logistic Regression (99.82%) and Classic Random Forest *Unregularized* (99.63%, oob score). Regarding to the **PCA** Logistic Regression, this model was trained on a reduced version of our original dateset. Thus, our model is now become a "black box" (we don't know which genes are the most important to determining a cancer category). Conversely, the Classic Random Forest *(Unregularized)* was trained on **20531** features. Therefore, this element makes our Random Forest much impractical. Our gooal should be to come up with a regularized "white box" model which reacheas the highest accuracy possible

Firstly, we could try to should understand what are the most important features in our Random forest calssfier

In [13]:
features_percentage_arr = random_clf.feature_importances_
indices = np.where(features_percentage_arr > 0.003)[0]
top_features_names = random_clf.feature_names_in_[indices]
top_features_names.shape

(67,)

If we set a treshold of 0.003 all the features which have an importance bigger than 0.003 will be saved

Secondly, we could repeat the same process building upon the previous Logistic Regresion model. While standard Logistic regression performed great when predicting cancer classes we could now experiment using an Elastic Net regularization. Also, the goal is to find the best trade-off ratio between feature reduction and predictive performance (specifically "C" and "l1_ratio") using RandomSearchCV

In [14]:
from sklearn.svm import l1_min_c
c_min = l1_min_c(X_train, y_train, loss="log")
c_min

np.float64(0.0002650961711777969)

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV

pip = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="saga", 
                       l1_ratio=1, 
                       tol=1e-2, 
                       max_iter=500,
                       random_state=42))

params = {"logisticregression__l1_ratio": [0.6, 0.7, 0.8, 0.9],
         "logisticregression__C": np.geomspace(c_min * 2, 0.8, num=10)}

random_search = RandomizedSearchCV(pip, param_distributions=params, 
                                 cv=3, n_jobs=-1, scoring="accuracy")

random_search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step..., tol=0.01))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'logisticregression__C': array([5.3019...00000000e-01]), 'logisticregression__l1_ratio': [0.6, 0.7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_gr

In [16]:
random_search.best_params_

{'logisticregression__l1_ratio': 0.8,
 'logisticregression__C': np.float64(0.013714175068063727)}

In [17]:
random_search.best_score_

np.float64(0.9982078853046595)

In [18]:
elastic_net_log_reg = random_search.best_estimator_

In [19]:
joblib.dump(elastic_net_log_reg, os.path.join(models_dir, "elastic_net_log_reg"))

['C:\\Users\\admin\\OneDrive\\Desktop\\Python\\cancer_classification\\models\\elastic_net_log_reg']

In [20]:
from sklearn.linear_model import SGDClassifier

pip = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", penalty="elasticnet", 
                  early_stopping=True, random_state=42))

params = {"sgdclassifier__l1_ratio": [0.6, 0.7, 0.8, 0.9],
         "sgdclassifier__alpha": np.logspace(-4, -1, 10)}

random_search_sgd = RandomizedSearchCV(pip, param_distributions=params, 
                                 cv=2, n_jobs=-1, scoring="accuracy")

random_search_sgd.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'sgdclassifier__alpha': array([0.0001..., 0.1 ]), 'sgdclassifier__l1_ratio': [0.6, 0.7, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refi

In [21]:
random_search_sgd.best_params_

{'sgdclassifier__l1_ratio': 0.6,
 'sgdclassifier__alpha': np.float64(0.004641588833612777)}

In [22]:
random_search_sgd.best_score_

np.float64(0.9964285714285714)

In [23]:
elastic_net_sgd_clf = random_search_sgd.best_estimator_

In [24]:
joblib.dump(elastic_net_sgd_clf, os.path.join(models_dir, "elastic_net_sgd_log_reg"))

['C:\\Users\\admin\\OneDrive\\Desktop\\Python\\cancer_classification\\models\\elastic_net_sgd_log_reg']

After further evaluation we selected the 67 most important genes used by our Unregularized Random Forest (oob accuracy: **99.63%**) trained on the full original dataset. Our Elastic Net Logistic Regression reached an astonishing **99.81%** accuracy tested using cross validation using 3 folds. The **SGD Elastic Net Classfier** reached a **99.63%** accuracy score. 

In [25]:
# First, we must acces the log reg step in the pipeline. Then we can extract featurews weights
features_weights = elastic_net_log_reg.named_steps["logisticregression"].coef_

# Save the features weights array in memory to use it later
features_weights_dir_path = os.path.join(get_project_root(), "data", "features_weights")
arr_final_path = os.path.join(features_weights_dir_path, "numpy_features_weights_arr.npy")

np.save(arr_final_path, features_weights)
if os.path.isfile(arr_final_path):
    print(f"Array saved at: {arr_final_path}") 
else:
    print("Array File not found")
    
# Check if every feature have at least one non zero weight. Conversely, we can delete that feateure because it has no importance
rows_used_features_idx, columns_used_features_idx = np.where(features_weights !=0)
# The output is a tuple representing the indices of both rows and columns which meet teh !=0 condition.
# Actually, the indices of the columns also represents the indices of the most important original dataset features

best_features_indices = np.unique(columns_used_features_idx)

Array saved at: C:\Users\admin\OneDrive\Desktop\Python\cancer_classification\data\features_weights\numpy_features_weights_arr.npy


In [26]:
reduced_features_weights = features_weights[:, best_features_indices]

absolute_reduced_features_weights = np.absolute(reduced_features_weights)

feature_per_class_absolute_max_weight = np.max(absolute_reduced_features_weights, axis=0)

# This line yields an array containing the indices values for the columns of the original features_weights 2d array
# Which meet the condition (absolute_max - absolute_min) // 2
absolute_max = np.max(feature_per_class_absolute_max_weight)
absolute_min = np.min(feature_per_class_absolute_max_weight)
threshold = ((absolute_max - absolute_min) / 2)
best_reduced_features_idx = np.where(feature_per_class_absolute_max_weight > threshold)[0]

In [27]:
final_original_features_idx = best_features_indices[best_reduced_features_idx]
final_weights = features_weights[:, final_original_features_idx]

print(f"Final weights shape: {final_weights.shape}")
print(f"Original features indices to keep: {final_original_features_idx}")

Final weights shape: (5, 37)
Original features indices to keep: [  357  2037  2747  3523  3737  5407  5578  6530  6611  6748  6876  7964
  9175  9176  9652 10731 11352 11550 11903 13639 14092 15161 15577 15589
 15591 15894 15895 15896 15898 15899 15900 17801 17905 17993 18135 18746
 19542]


We successfully completed an outstanding **99.82% feature reduction** passing from **20531** to only **39** genes values (features). Now we can create a reduced version of our original dataset and we can evaluate a Logistic Regression model on that reduced dataset (see Notebook 03)

After performing dimensionality reduction and feature selection, the best 39 genes we identified with the highest predictive power for cancer calssificationn were the following:

In [31]:
X_train.columns[final_original_features_idx]

Index(['ADIPOQ|9370', 'C17orf73|55018', 'C8orf85|441376', 'CDX1|1044',
       'CHRNA2|1135', 'EFHD1|80303', 'EN1|2019', 'FLJ45983|399717',
       'FOXF1|2294', 'FZD5|7855', 'GATA3|2625', 'HNF1B|6928', 'KLK2|3817',
       'KLK3|354', 'LMX1B|4010', 'MCTP2|55784', 'MUC21|394263', 'NAPSA|9476',
       'NKX2-1|7080', 'PLUNC|51297', 'PRLR|5618', 'ROS1|6098', 'SCEL|8796',
       'SCGB2A2|4250', 'SCGB3A2|117156', 'SFTA2|389376', 'SFTA3|253970',
       'SFTPA1|653509', 'SFTPB|6439', 'SFTPC|6440', 'SFTPD|6441', 'TCF21|6943',
       'TFAP2B|7021', 'THRSP|7069', 'TMEFF2|23671', 'TRPS1|7227',
       'WDR69|164781'],
      dtype='str')